## Single Agent for RE and Design

In [1]:

!pip install google-adk -q
!pip install litellm -q


[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip
Installation complete.


In [34]:
# Import libraries
import os
from dotenv import load_dotenv
import asyncio
import uuid
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.genai import types

import warnings
# Ignore all warnings
warnings.filterwarnings("ignore")

import logging
logging.basicConfig(level=logging.ERROR)

In [35]:
# LLM API Keys

# Load env variables from system
load_dotenv()

GOOGLE_API_KEY = os.environ['GOOGLE_API_KEY']

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"


In [36]:
# Model constants
GPT_OSS = "ollama_chat/gpt-oss:20b"

In [37]:
SYSTEM_PROMPT = """You are an expert software architect specializing in analyzing requirements and designing software.

## Workflow
When a user requests to design a software system:
1. **Document** the requirement in detail.
2. **Analyze** the requirements.
3. **Generate** a software requirements specification (SRS) document with detailed requirements.
4. **Design** the system using an architecture diagram and detailed class and sequence diagrams using mermaid notations.
5. **Present** the SRS and design documents to the user.
"""

In [42]:
# Create the agent
requirements_design_agent = Agent(
    name="requirements_design_agent",
    description="A requirements and design agent.",
    model="gemini-2.5-flash",
    instruction=SYSTEM_PROMPT
)
print(f"Agent '{requirements_design_agent.name}' created using model '{GPT_OSS}'.")

Agent 'requirements_design_agent' created using model 'ollama_chat/gpt-oss:20b'.


In [43]:
# Setup Session Service and Runner

session_service = InMemorySessionService()

APP_NAME = "requirements_design_app"
USER_ID = str(uuid.uuid4())
SESSION_ID = str(uuid.uuid4())

# Create a conversational session
session = await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID
)
print(f"Session created: App='{APP_NAME}', User='{USER_ID}', Session='{SESSION_ID}'")

# Create the runner to execute the agent loop
runner = Runner(
    agent=requirements_design_agent,
    app_name=APP_NAME,
    session_service=session_service
)
print(f"Runner created for agent '{runner.agent.name}'.")

Session created: App='requirements_design_app', User='268b8285-e13e-4c5b-82c9-1a6e59e2d886', Session='48819a1d-e38c-42a6-a9df-ffc36bbb4d95'
Runner created for agent 'requirements_design_agent'.


In [44]:
# Async function to call agent and retrieve final response.

from google.genai import types

async def execute_agent(query: str):
  """Sends a user's input to the agent and retrieves the response."""
  print(f"\n>>> User Query: {query}")

  # Format message
  content = types.Content(role='user', parts=[types.Part(text=query)])

  response = "Sorry, no response."

  async for event in runner.run_async(user_id=USER_ID, session_id=SESSION_ID, new_message=content):
      if event.is_final_response():
          if event.content and event.content.parts:
             response = event.content.parts[0].text
          elif event.actions and event.actions.escalate:
             response = f"Agent escalated: {event.error_message or 'No specific message.'}"
          break

  print(f"<<< Response from the RE and Design Agent: {response}")

In [45]:
# Test agent using sample requests.

async def execute_turns():
    await execute_agent("Design a tic-tac-toe game.")

await execute_turns()


>>> User Query: Design a tic-tac-toe game.
<<< Response from the RE and Design Agent: Okay, I will design a Tic-Tac-Toe game based on your request.

## 1. Document Requirements

The user wants to design a "Tic-Tac-Toe game."

## 2. Analyze Requirements

This is a standard 2-player game on a 3x3 grid.
Key functionalities include:
*   **Game Setup**: Initialize a 3x3 board.
*   **Player Turns**: Alternate turns between two players (typically 'X' and 'O').
*   **Move Input**: Players choose an empty cell to place their marker.
*   **Move Validation**: Ensure the chosen cell is valid and empty.
*   **Win Condition**: Detect when a player has three of their markers in a row, column, or diagonal.
*   **Draw Condition**: Detect when the board is full and no player has won.
*   **Game End**: Announce the winner or a draw.
*   **User Interface**: Display the board and messages to the players.
*   **Replay**: Option to start a new game.

For simplicity, we'll assume a console-based game, but th